# DIYANG RADITYA ANWAR

# SKRIPSI : Identifikasi Peluang Optimasi Biaya Layanan Cloud AWS PT Jalin Mayantara Menggunakan Random Forest dan LLM

# 1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error, r2_score

# 2. Data Loading & Understanding

In [2]:
data = pd.read_csv("../Datasets/original-copy.csv")
data.head(10)

In [3]:
print(f"Ukuran baris x kolom : {data.shape}")

In [4]:
data.info(verbose=True)

In [5]:
data.isna().sum()

# 3. Data Cleaning
## Fase cleaning 1

Dataset log operasional server PT Jayantara pada bulan Februari 2025 ini memiliki 1343347 baris dan 188 kolom. Namun, masih banyak yang kotor datanya seperti data yang banyak null valuesnya, format yang belum sesuai, dll. Bisa kita drop terlebih dahulu fitur-fitur yang semuanya null. Fitur-fitur tersebut adalah :
- bill_invoice_id
- product_datatransferout
- product_pre_installed_sw
- reservation_end_time
- reservation_modification_status
- reservation_normalized_units_per_reservation
- reservation_number_of_reservations
- reservation_start_time
- reservation_total_reserved_normalized_units
- reservation_total_reserved_units
- reservation_units_per_reservation

In [6]:
data.drop(columns=["bill_invoice_id", "product_datatransferout", "product_pre_installed_sw", "reservation_end_time",
                    "reservation_modification_status", "reservation_normalized_units_per_reservation", "reservation_number_of_reservations", 
                    "reservation_start_time", "reservation_total_reserved_normalized_units", "reservation_total_reserved_units", 
                    "reservation_units_per_reservation"], axis=1, inplace=True)
data.head(10)

In [7]:
data2 = data.copy()

In [8]:
998327/1343347

In [9]:
data2["product_usagetype"].value_counts()

In [10]:
# 724129/1343347
# data2["product_region"].isna().sum()
# data2["product_product_family"].value_counts()
data2["line_item_usage_type"].value_counts()

In [11]:
data2["product_region"].value_counts()

### BUKTI `product_availability` tidak diperlukan karena korelasinya kecil dgn target variabel

In [12]:
df_analysis = data2[['product_availability', 'line_item_unblended_cost']].copy()

# Remove rows where availability is missing (NaN)
df_analysis = df_analysis.dropna(subset=['product_availability'])

# Convert '99.99%' string to float 99.99
# We strip the '%' sign and convert to a number
df_analysis['availability_numeric'] = df_analysis['product_availability'].astype(str).str.rstrip('%').astype(float)

# Ensure cost is numeric (just in case)
df_analysis['line_item_unblended_cost'] = pd.to_numeric(df_analysis['line_item_unblended_cost'], errors='coerce')


# 2. CHECKING CORRELATION
# A score close to 1.0 means strong positive relationship (Higher SLA = Higher Cost)
correlation = df_analysis['availability_numeric'].corr(df_analysis['line_item_unblended_cost'])

print(f"Correlation between Availability and Cost: {correlation:.4f}")


# 3. VISUALIZATION (Boxplot)
# This shows the range of prices for each availability tier
plt.figure(figsize=(10, 6))
sns.boxplot(x='product_availability', y='line_item_unblended_cost', data=df_analysis)

plt.title('Cost Distribution by Product Availability SLA')
plt.xlabel('Availability Guarantee')
plt.ylabel('Unblended Cost (USD)')
plt.yscale('log') # Log scale helps because cloud costs vary wildly (cents vs thousands)
plt.grid(True, which="both", ls="--", alpha=0.3)
plt.show()

In [13]:
data2.isna().sum()

Setelah melakukan pemahaman data dan data cleaning untuk menggunakan kolom-kolom yang mana saja. Berikut adalah list-list fitur yang akan dipakai.

**DEPENDENT VARIABLE** : line_item_unblended_cost

**INDEPENDENT VARIABLE**:

**A. Usage and Specs**

1) line_item_usage_amount
2) line_item_normalization_factor
3) line_item_normalized_usage_amount
4) product_clock_speed
5) line_item_line_item_type
6) line_item_usage_type
7) line_item_operation

**B. Location and Network**

8) product_region
9) line_item_availability_zone
10) product_to_location
11) product_from_location
12) product_transfer_type

**C. Product Metadata**

13) line_item_product_code
14) product_product_family
15) pricing_term
16) pricing_unit

**D. Business Tags**

17) resource_tags_user_customer
18) resource_tags_user_project
19) resource_tags_user_project_owner
20) resource_tags_user_tech_owner

**E. Time**

21) line_item_usage_start_date
22) line_item_usage_end_date

--> menghasilkan kolom duration hours (dr 2 kolom atas), day_of_week (dari start_date), hour_of_day (dari start_date)

semua log catatan cleaning bagian ini terdapat pada file `cleaning-log-phase1.txt`

In [14]:
df_cleaned1 = data2[["line_item_usage_start_date", "line_item_usage_end_date", "line_item_usage_amount", 
                     "line_item_normalization_factor", "line_item_normalized_usage_amount", "product_clock_speed", 
                     "line_item_line_item_type", "line_item_usage_type", "line_item_operation", 
                     "product_region", "line_item_availability_zone", "product_to_location", "product_from_location", 
                     "product_transfer_type", "line_item_product_code", "product_product_family", "pricing_term", 
                     "pricing_unit", "resource_tags_user_customer", "resource_tags_user_project", 
                     "resource_tags_user_project_owner", "resource_tags_user_tech_owner", "line_item_unblended_cost"]].copy()
df_cleaned1

In [15]:
df_cleaned1.info()

Namun, setelah di pilih-pilih fitur yang penting ada beberapa fitur yang memiliki missing values yang sangat besar bahkan melebihi 1 juta rows, yaitu `product_clock_speed` dan `line_item_availability_zone`. Sehingga harus di drop kolomnya untuk menghindari model ML tidak bisa belajar karena model seperti Random Forest tidak bisa belajar dari `null` values dan kurangnya informasi aktual

In [16]:
df_cleaned1.isna().sum()

In [17]:
df_cleaned1.drop(columns=["line_item_availability_zone", "product_clock_speed"], inplace=True)
df_cleaned1.isna().sum()

In [18]:
df_cleaned1.head(20)

In [19]:
df_cleaned1.shape

In [20]:
df_cleaned1.info()

## Fase cleaning 2
mengisi value yang null dari fitur-fitur yang akan dipakai dan konversi tipe data menjadi yang semestinya

### 1. Mengisi nilai-nilai yang hilang secara "struktural" 

Grup A : kolom-kolom transfer data (724k missing values)
- `product_to_location`
- `product_from_location`
- `product_transfer_type`

Aksi eksekusi : mengisi dengan value "No Transfer" karena kolom-kolom ini hanya terisi ketika data secara fisik berpindah melalui jaringan.

In [21]:
# Fill Transfer info
cols_transfer = ['product_to_location', 'product_from_location', 'product_transfer_type']
df_cleaned1[cols_transfer] = df_cleaned1[cols_transfer].fillna('No Transfer')

Grup B : kolom-kolom resource tags (100k-500k misssing)
- resource_tags_user_customer
- resource_tags_user_project
- resource_tags_user_project_owner
- resource_tags_user_tech_owner

Aksi eksekusi : mengisi dengan value "Untagged" karena resource-resource pada kolom tersebut sumber data yang tidak dilabeli

In [22]:
cols_tags = [
    'resource_tags_user_customer',
    'resource_tags_user_project',
    'resource_tags_user_project_owner',
    'resource_tags_user_tech_owner'
]
df_cleaned1[cols_tags] = df_cleaned1[cols_tags].fillna('Untagged')

Grup C: kolom `pricing_term`

Aksi eksekusi : mengisi dengan value "OnDemand" karena dalam banyak laporan AWS, label "On-Demand" diperlakukan sebagai status null default. Jika sebuah baris tidak Reserved dan tidak Spot, maka secara definisi, baris tersebut adalah On-Demand.

In [23]:
df_cleaned1['pricing_term'] = df_cleaned1['pricing_term'].fillna('OnDemand')

### 2. Drop baris-baris yang random missing values

Grup D : kolom-kolom yang missing valuesnya sedikit
- `line_item_operation` (14k missing)
- `product_product_family` (20k missing)
- `product_region` (367 missing)
- `pricing_unit` (367 missing)

Aksi eksekusi : drop null values, tidak masalah kalau drop hanya 20k+- baris

In [24]:
# DROP Rows with Random Missing Values
# Identify columns where can't afford to guess
cols_drop_rows = [
    'line_item_operation',
    'product_product_family',
    'product_region',
    'pricing_unit'
]

df_cleaned1.dropna(subset=cols_drop_rows, inplace=True)

In [25]:
df_cleaned1.isna().sum()

# 4. Feature Engineering

konversi kolom `line_item_usage_start_date` dan `line_item_usage_end_date` menjadi tipe data datetime, ekstraksi fitur Duration, Day of month, and Day/Hour, dan drop kolom `line_item_usage_start_date` dan `line_item_usage_end_date`

In [26]:
# 1. Convert to datetime objects first
df_cleaned1['line_item_usage_start_date'] = pd.to_datetime(df_cleaned1['line_item_usage_start_date'])
df_cleaned1['line_item_usage_end_date'] = pd.to_datetime(df_cleaned1['line_item_usage_end_date'])

# 2. Extract The "Who, When, How Long"

# A. The Specific Date (1, 2, ... 28) -> "Each single day"
df_cleaned1['day_of_month'] = df_cleaned1['line_item_usage_start_date'].dt.day

# B. The Day Name (0=Monday, 6=Sunday) -> "What day it is"
df_cleaned1['day_of_week'] = df_cleaned1['line_item_usage_start_date'].dt.dayofweek

# C. The Time of Day (0-23)
df_cleaned1['hour_of_day'] = df_cleaned1['line_item_usage_start_date'].dt.hour

# D. The Duration (How long did it run?)
# We convert the timedelta to hours (float)
df_cleaned1['duration_hours'] = (df_cleaned1['line_item_usage_end_date'] - df_cleaned1['line_item_usage_start_date']).dt.total_seconds() / 3600

# 3. NOW drop the original date columns safely
df_cleaned1.drop(['line_item_usage_start_date', 'line_item_usage_end_date'], axis=1, inplace=True)

# ---------------------------------------------------------
# Verification
# This will show you the new columns with values like 1, 15, 28 for days
print(df_cleaned1[['day_of_month', 'day_of_week', 'hour_of_day']].head())
print("-" * 30)
print(df_cleaned1.info())

In [27]:
df_cleaned1.drop(["duration_hours"], axis=1, inplace=True)
df_cleaned1

# 5. Data preprocessing and Modelling
Melakukan sortir dataset berdasarkan waktu terlebih dahulu


In [28]:
df_cleaned1.sort_values(by=['day_of_month', 'hour_of_day'], inplace=True)

# Reset index after sorting so it's clean 0 to 1,343,346
df_cleaned1.reset_index(drop=True, inplace=True)
df_cleaned1

Menstandarisasi kolom `resource_tags_user_customer` terlebih dahulu yang kemendikbud-nya memiliki beberapa jenis

In [29]:
df_cleaned1['resource_tags_user_customer'] = df_cleaned1['resource_tags_user_customer'].str.strip()

replace_dict = {
    'Kemdikbud': 'Kemendikbud',
    'Kemendikbudristek': 'Kemendikbud',
    'Kemdikbudristek': 'Kemendikbud',
    'Kemdukbud': 'Kemendikbud'
}

df_cleaned1['resource_tags_user_customer'] = df_cleaned1['resource_tags_user_customer'].replace(replace_dict)

print("Variasi nama customer setelah distandarisasi:")
df_cleaned1['resource_tags_user_customer'].value_counts()

Sebelum beralih ke data preprocessing, berikut adalah fitur-fitur yang akan dipakai untuk pengembangan machine learning. 

Variabel independen adalah `line_item_unblended_cost`.

Variabel dependen terdapat 21 variabel

In [30]:
df_cleaned1.head(20)

In [33]:
list(df_cleaned1.columns)

List jumlah types of values per categorical column sebelum encoding
- line_item_line_item_type -> 3 tipe
- line_item_usage_type -> 554 tipe
- line_item_operation -> 136 tipe
- product_region -> 21 tipe
- product_to_location -> 52 tipe
- product_from_location -> 27 tipe
- product_transfer_type -> 8 tipe
- line_item_product_code -> 23 tipe
- product_product_family -> 28 tipe
- pricing_term -> 2 tipe
- pricing_unit -> 26 tipe
- resource_tags_user_customer -> 11 tipe
- resource_tags_user_project -> 56 tipe 
- resource_tags_user_project_owner -> 6 tipe
- resource_tags_user_tech_owner -> 7 tipe

Untuk encoding kolom-kolom kategorikal di atas beserta jumlah tipe datanya, menggunakan frequency encoding untuk kolom yang high cardinality (>= 10 tipe):
- `line_item_usage_type` 
- `line_item_operation` 
- `resource_tags_user_project` 
- `product_to_location` 
- `product_product_family` 
- `product_from_location` 
- `pricing_unit`
- `line_item_product_code` 
- `product_region` 
- `resource_tags_user_customer`

dan menggunakan OHE untuk yang low cardinality (<10 tipe)
- `pricing_term`
- `line_item_line_item_type`
- `project_owner`
- `tech_owner`
- `transfer_type`

In [133]:
cols_ohe = [
    'pricing_term',
    'line_item_line_item_type',
    'resource_tags_user_project_owner',
    'resource_tags_user_tech_owner',
    'product_transfer_type'
]

cols_freq = [
    'line_item_usage_type', 'line_item_operation', 'resource_tags_user_project',
    'product_to_location', 'product_product_family', 'product_from_location',
    'pricing_unit', 'line_item_product_code', 'product_region',
    'resource_tags_user_customer'
]

# Dictionary untuk menyimpan peta Frequency Encoding (penting buat streamlit)
freq_maps = {}

In [134]:
for col in cols_freq:
    df_cleaned1[col] = df_cleaned1[col].astype(str) # Pastikan string
    freq_encoding = df_cleaned1[col].value_counts(normalize=True).to_dict() # Hitung %
    df_cleaned1[col] = df_cleaned1[col].map(freq_encoding) # Replace teks dengan angka %
    freq_maps[col] = freq_encoding # Simpan kamusnya

# C. Eksekusi One-Hot Encoding
df_ohe = pd.get_dummies(df_cleaned1, columns=cols_ohe, dtype=float)

# Simpan nama kolom final agar urutan di Streamlit sama persis
feature_names = df_ohe.drop(columns=['line_item_unblended_cost']).columns.tolist()

print(f"✅ Encoding Selesai. Total Fitur: {len(feature_names)}")

### Data Splitting based on time (1-16 February 2025)

In [135]:
cutoff_date = 14

# Filter data dari df_ohe
train_data = df_ohe[df_ohe['day_of_month'] < cutoff_date]
test_data = df_ohe[df_ohe['day_of_month'] >= cutoff_date]

target = 'line_item_unblended_cost'

# Drop target dari df_ohe untuk jadi X
X_train = train_data.drop(columns=[target])
y_train = train_data[target]

X_test = test_data.drop(columns=[target])
y_test = test_data[target]

print(f"Siap Training! Fitur yang masuk: {X_train.shape[1]} kolom")

In [136]:
print("-" * 30)
print(f"SPLIT DATA (Total 16 Hari)")
print(f"Data Training (Tgl 1-13): {X_train.shape[0]} baris")
print(f"Data Testing  (Tgl 14-16): {X_test.shape[0]} baris")
print("-" * 30)

### Training and Evaluation

In [138]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"📊 HASIL EVALUASI:")
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2 Score: {r2:.4f}")
print("-" * 30)

Untuk membuktikan hasil model prediksi tersebut adalah valid, coba memeriksa feature importance dan Scatter Plot

In [139]:
importances = rf_model.feature_importances_
feature_names = X_train.columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='viridis')
plt.title('Top 10 Cost Drivers (Feature Importance)')
plt.show()

# 2. Cek Sebaran Error (Residuals)
# Kita lihat apakah model cuma jago di angka 0 tapi hancur di angka besar.

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Garis diagonal sempurna
plt.xlabel('Actual Cost (USD)')
plt.ylabel('Predicted Cost (USD)')
plt.title('Actual vs Predicted Cost')
plt.show()

### Validasi Model: Feature Importance Analysis

Mengingat nilai akurasi ($R^2$) yang sangat tinggi (0.9999), validasi logika diperlukan untuk memastikan tidak terjadi *Data Leakage* (kebocoran data).

Berdasarkan grafik **Feature Importance** di atas, fitur **`line_item_usage_amount`** teridentifikasi sebagai fitur paling dominan (skor > 0.8). Hal ini membuktikan bahwa model telah mempelajari kausalitas yang benar secara bisnis Cloud Billing, di mana:

$$\text{Cost} \approx \text{Usage Amount} \times \text{Rate}$$

Dominasi fitur penggunaan (`usage`) dibanding fitur identitas atau fitur administratif lainnya mengonfirmasi bahwa model memprediksi biaya berdasarkan **volume konsumsi sumber daya**, bukan menghafal pola data yang tidak relevan (kebetulan).